# 🐧 Esercitazione 3 — Ensemble, KNN e clustering

**new ALF – Tutorato** · Corso di Laurea in Informatica · AA 2026/27

Nell'Esercitazione 2 abbiamo scelto un albero con la cross-validation e `GridSearchCV`. Oggi partiamo da lì e vediamo tre cose nuove: gli **ensemble**, che mettono insieme tanti alberi (**random forest** e **boosting**); il **KNN**, che classifica guardando i pinguini più vicini; e il **clustering** con **k-means**, che cerca gruppi nei dati **senza etichette**. KNN e k-means ragionano con le **distanze**, e per questo in entrambi conta la **scala delle feature**.

Useremo di nuovo il dataset **Palmer Penguins**: le misure di 344 pinguini di tre specie (Adelie, Chinstrap e Gentoo), raccolte tra il 2007 e il 2009 su tre isole dell'arcipelago Palmer, in Antartide.


> ### ⚠️ Prima di tutto: fai la tua copia
> Questo notebook è condiviso in **sola lettura**: puoi eseguirlo e modificarlo, ma le modifiche **non verranno salvate**.
> Vai su **File → Salva una copia in Drive**. Si aprirà una nuova scheda con la *tua* copia (la trovi poi su Google Drive, nella cartella *Colab Notebooks*): lavora lì.

### Come si usa questo notebook
- Ogni **cella** si esegue con **Shift + Invio** (o con il tasto ▶ a sinistra). Esegui le celle **in ordine**, dall'alto verso il basso.
- Le celle segnate con **✏️** sono esercizi: modifica il codice dove indicato. Sotto ogni esercizio c'è un blocco **💡 Soluzione** chiuso: prova prima da solo. Le celle successive non vanno in errore se lasci un esercizio com'è, ma i grafici e le tabelle che dipendono dall'esercizio restano vuoti: **prima di proseguire copia la soluzione** nella cella dell'esercizio ed eseguila. Vale soprattutto per il benchmark della 2.5, da cui dipende la valutazione finale.
- I box **🐍** spiegano le cose nuove di Python che incontriamo strada facendo.
- Le sezioni segnate con **⏩** sono approfondimenti.
- I box **📋** dicono quale problema stiamo risolvendo. I dati sono sempre gli stessi pinguini: cambia la **domanda**, e con lei il target `y`, le feature `X` e il modo di valutare il modello.
- I box **🤔** sono domande su cui riflettere (spesso con la risposta chiusa sotto), i **📌** i concetti chiave.
- **🔒 / 🔓**: il test set serve a **valutare il modello finale**. Quando dobbiamo **scegliere** un modello resta chiuso, e si apre una volta sola, alla fine.
- Se qualcosa si rompe: menu **Runtime → Riavvia sessione ed esegui tutto** (*Restart session and run all*).

In [ ]:
#@title ⚙️ Setup: esegui questa cella per prima (crea il file con i dati)
# Questa cella scrive nel workspace di Colab il file "pinguini.csv". Puoi vederlo nel pannello 📁 a sinistra.
# Nota: il workspace di Colab si svuota quando la sessione termina,
# quindi questa cella va rieseguita ogni volta che riapri il notebook.
#
# Dati: Palmer Penguins (Gorman, Williams & Fraser, 2014), raccolti alla Palmer Station (Antartide)
# e distribuiti con licenza CC0 dal pacchetto palmerpenguins di A. Horst, A. Hill e K. Gorman.
# Nomi delle colonne e valori del sesso tradotti in italiano.

PINGUINI_CSV = """specie,isola,becco_lunghezza,becco_profondita,pinna,peso,sesso
Adelie,Torgersen,39.1,18.7,181,3750,maschio
Adelie,Torgersen,39.5,17.4,186,3800,femmina
Adelie,Torgersen,40.3,18,195,3250,femmina
Adelie,Torgersen,,,,,
Adelie,Torgersen,36.7,19.3,193,3450,femmina
Adelie,Torgersen,39.3,20.6,190,3650,maschio
Adelie,Torgersen,38.9,17.8,181,3625,femmina
Adelie,Torgersen,39.2,19.6,195,4675,maschio
Adelie,Torgersen,34.1,18.1,193,3475,
Adelie,Torgersen,42,20.2,190,4250,
Adelie,Torgersen,37.8,17.1,186,3300,
Adelie,Torgersen,37.8,17.3,180,3700,
Adelie,Torgersen,41.1,17.6,182,3200,femmina
Adelie,Torgersen,38.6,21.2,191,3800,maschio
Adelie,Torgersen,34.6,21.1,198,4400,maschio
Adelie,Torgersen,36.6,17.8,185,3700,femmina
Adelie,Torgersen,38.7,19,195,3450,femmina
Adelie,Torgersen,42.5,20.7,197,4500,maschio
Adelie,Torgersen,34.4,18.4,184,3325,femmina
Adelie,Torgersen,46,21.5,194,4200,maschio
Adelie,Biscoe,37.8,18.3,174,3400,femmina
Adelie,Biscoe,37.7,18.7,180,3600,maschio
Adelie,Biscoe,35.9,19.2,189,3800,femmina
Adelie,Biscoe,38.2,18.1,185,3950,maschio
Adelie,Biscoe,38.8,17.2,180,3800,maschio
Adelie,Biscoe,35.3,18.9,187,3800,femmina
Adelie,Biscoe,40.6,18.6,183,3550,maschio
Adelie,Biscoe,40.5,17.9,187,3200,femmina
Adelie,Biscoe,37.9,18.6,172,3150,femmina
Adelie,Biscoe,40.5,18.9,180,3950,maschio
Adelie,Dream,39.5,16.7,178,3250,femmina
Adelie,Dream,37.2,18.1,178,3900,maschio
Adelie,Dream,39.5,17.8,188,3300,femmina
Adelie,Dream,40.9,18.9,184,3900,maschio
Adelie,Dream,36.4,17,195,3325,femmina
Adelie,Dream,39.2,21.1,196,4150,maschio
Adelie,Dream,38.8,20,190,3950,maschio
Adelie,Dream,42.2,18.5,180,3550,femmina
Adelie,Dream,37.6,19.3,181,3300,femmina
Adelie,Dream,39.8,19.1,184,4650,maschio
Adelie,Dream,36.5,18,182,3150,femmina
Adelie,Dream,40.8,18.4,195,3900,maschio
Adelie,Dream,36,18.5,186,3100,femmina
Adelie,Dream,44.1,19.7,196,4400,maschio
Adelie,Dream,37,16.9,185,3000,femmina
Adelie,Dream,39.6,18.8,190,4600,maschio
Adelie,Dream,41.1,19,182,3425,maschio
Adelie,Dream,37.5,18.9,179,2975,
Adelie,Dream,36,17.9,190,3450,femmina
Adelie,Dream,42.3,21.2,191,4150,maschio
Adelie,Biscoe,39.6,17.7,186,3500,femmina
Adelie,Biscoe,40.1,18.9,188,4300,maschio
Adelie,Biscoe,35,17.9,190,3450,femmina
Adelie,Biscoe,42,19.5,200,4050,maschio
Adelie,Biscoe,34.5,18.1,187,2900,femmina
Adelie,Biscoe,41.4,18.6,191,3700,maschio
Adelie,Biscoe,39,17.5,186,3550,femmina
Adelie,Biscoe,40.6,18.8,193,3800,maschio
Adelie,Biscoe,36.5,16.6,181,2850,femmina
Adelie,Biscoe,37.6,19.1,194,3750,maschio
Adelie,Biscoe,35.7,16.9,185,3150,femmina
Adelie,Biscoe,41.3,21.1,195,4400,maschio
Adelie,Biscoe,37.6,17,185,3600,femmina
Adelie,Biscoe,41.1,18.2,192,4050,maschio
Adelie,Biscoe,36.4,17.1,184,2850,femmina
Adelie,Biscoe,41.6,18,192,3950,maschio
Adelie,Biscoe,35.5,16.2,195,3350,femmina
Adelie,Biscoe,41.1,19.1,188,4100,maschio
Adelie,Torgersen,35.9,16.6,190,3050,femmina
Adelie,Torgersen,41.8,19.4,198,4450,maschio
Adelie,Torgersen,33.5,19,190,3600,femmina
Adelie,Torgersen,39.7,18.4,190,3900,maschio
Adelie,Torgersen,39.6,17.2,196,3550,femmina
Adelie,Torgersen,45.8,18.9,197,4150,maschio
Adelie,Torgersen,35.5,17.5,190,3700,femmina
Adelie,Torgersen,42.8,18.5,195,4250,maschio
Adelie,Torgersen,40.9,16.8,191,3700,femmina
Adelie,Torgersen,37.2,19.4,184,3900,maschio
Adelie,Torgersen,36.2,16.1,187,3550,femmina
Adelie,Torgersen,42.1,19.1,195,4000,maschio
Adelie,Torgersen,34.6,17.2,189,3200,femmina
Adelie,Torgersen,42.9,17.6,196,4700,maschio
Adelie,Torgersen,36.7,18.8,187,3800,femmina
Adelie,Torgersen,35.1,19.4,193,4200,maschio
Adelie,Dream,37.3,17.8,191,3350,femmina
Adelie,Dream,41.3,20.3,194,3550,maschio
Adelie,Dream,36.3,19.5,190,3800,maschio
Adelie,Dream,36.9,18.6,189,3500,femmina
Adelie,Dream,38.3,19.2,189,3950,maschio
Adelie,Dream,38.9,18.8,190,3600,femmina
Adelie,Dream,35.7,18,202,3550,femmina
Adelie,Dream,41.1,18.1,205,4300,maschio
Adelie,Dream,34,17.1,185,3400,femmina
Adelie,Dream,39.6,18.1,186,4450,maschio
Adelie,Dream,36.2,17.3,187,3300,femmina
Adelie,Dream,40.8,18.9,208,4300,maschio
Adelie,Dream,38.1,18.6,190,3700,femmina
Adelie,Dream,40.3,18.5,196,4350,maschio
Adelie,Dream,33.1,16.1,178,2900,femmina
Adelie,Dream,43.2,18.5,192,4100,maschio
Adelie,Biscoe,35,17.9,192,3725,femmina
Adelie,Biscoe,41,20,203,4725,maschio
Adelie,Biscoe,37.7,16,183,3075,femmina
Adelie,Biscoe,37.8,20,190,4250,maschio
Adelie,Biscoe,37.9,18.6,193,2925,femmina
Adelie,Biscoe,39.7,18.9,184,3550,maschio
Adelie,Biscoe,38.6,17.2,199,3750,femmina
Adelie,Biscoe,38.2,20,190,3900,maschio
Adelie,Biscoe,38.1,17,181,3175,femmina
Adelie,Biscoe,43.2,19,197,4775,maschio
Adelie,Biscoe,38.1,16.5,198,3825,femmina
Adelie,Biscoe,45.6,20.3,191,4600,maschio
Adelie,Biscoe,39.7,17.7,193,3200,femmina
Adelie,Biscoe,42.2,19.5,197,4275,maschio
Adelie,Biscoe,39.6,20.7,191,3900,femmina
Adelie,Biscoe,42.7,18.3,196,4075,maschio
Adelie,Torgersen,38.6,17,188,2900,femmina
Adelie,Torgersen,37.3,20.5,199,3775,maschio
Adelie,Torgersen,35.7,17,189,3350,femmina
Adelie,Torgersen,41.1,18.6,189,3325,maschio
Adelie,Torgersen,36.2,17.2,187,3150,femmina
Adelie,Torgersen,37.7,19.8,198,3500,maschio
Adelie,Torgersen,40.2,17,176,3450,femmina
Adelie,Torgersen,41.4,18.5,202,3875,maschio
Adelie,Torgersen,35.2,15.9,186,3050,femmina
Adelie,Torgersen,40.6,19,199,4000,maschio
Adelie,Torgersen,38.8,17.6,191,3275,femmina
Adelie,Torgersen,41.5,18.3,195,4300,maschio
Adelie,Torgersen,39,17.1,191,3050,femmina
Adelie,Torgersen,44.1,18,210,4000,maschio
Adelie,Torgersen,38.5,17.9,190,3325,femmina
Adelie,Torgersen,43.1,19.2,197,3500,maschio
Adelie,Dream,36.8,18.5,193,3500,femmina
Adelie,Dream,37.5,18.5,199,4475,maschio
Adelie,Dream,38.1,17.6,187,3425,femmina
Adelie,Dream,41.1,17.5,190,3900,maschio
Adelie,Dream,35.6,17.5,191,3175,femmina
Adelie,Dream,40.2,20.1,200,3975,maschio
Adelie,Dream,37,16.5,185,3400,femmina
Adelie,Dream,39.7,17.9,193,4250,maschio
Adelie,Dream,40.2,17.1,193,3400,femmina
Adelie,Dream,40.6,17.2,187,3475,maschio
Adelie,Dream,32.1,15.5,188,3050,femmina
Adelie,Dream,40.7,17,190,3725,maschio
Adelie,Dream,37.3,16.8,192,3000,femmina
Adelie,Dream,39,18.7,185,3650,maschio
Adelie,Dream,39.2,18.6,190,4250,maschio
Adelie,Dream,36.6,18.4,184,3475,femmina
Adelie,Dream,36,17.8,195,3450,femmina
Adelie,Dream,37.8,18.1,193,3750,maschio
Adelie,Dream,36,17.1,187,3700,femmina
Adelie,Dream,41.5,18.5,201,4000,maschio
Chinstrap,Dream,46.5,17.9,192,3500,femmina
Chinstrap,Dream,50,19.5,196,3900,maschio
Chinstrap,Dream,51.3,19.2,193,3650,maschio
Chinstrap,Dream,45.4,18.7,188,3525,femmina
Chinstrap,Dream,52.7,19.8,197,3725,maschio
Chinstrap,Dream,45.2,17.8,198,3950,femmina
Chinstrap,Dream,46.1,18.2,178,3250,femmina
Chinstrap,Dream,51.3,18.2,197,3750,maschio
Chinstrap,Dream,46,18.9,195,4150,femmina
Chinstrap,Dream,51.3,19.9,198,3700,maschio
Chinstrap,Dream,46.6,17.8,193,3800,femmina
Chinstrap,Dream,51.7,20.3,194,3775,maschio
Chinstrap,Dream,47,17.3,185,3700,femmina
Chinstrap,Dream,52,18.1,201,4050,maschio
Chinstrap,Dream,45.9,17.1,190,3575,femmina
Chinstrap,Dream,50.5,19.6,201,4050,maschio
Chinstrap,Dream,50.3,20,197,3300,maschio
Chinstrap,Dream,58,17.8,181,3700,femmina
Chinstrap,Dream,46.4,18.6,190,3450,femmina
Chinstrap,Dream,49.2,18.2,195,4400,maschio
Chinstrap,Dream,42.4,17.3,181,3600,femmina
Chinstrap,Dream,48.5,17.5,191,3400,maschio
Chinstrap,Dream,43.2,16.6,187,2900,femmina
Chinstrap,Dream,50.6,19.4,193,3800,maschio
Chinstrap,Dream,46.7,17.9,195,3300,femmina
Chinstrap,Dream,52,19,197,4150,maschio
Chinstrap,Dream,50.5,18.4,200,3400,femmina
Chinstrap,Dream,49.5,19,200,3800,maschio
Chinstrap,Dream,46.4,17.8,191,3700,femmina
Chinstrap,Dream,52.8,20,205,4550,maschio
Chinstrap,Dream,40.9,16.6,187,3200,femmina
Chinstrap,Dream,54.2,20.8,201,4300,maschio
Chinstrap,Dream,42.5,16.7,187,3350,femmina
Chinstrap,Dream,51,18.8,203,4100,maschio
Chinstrap,Dream,49.7,18.6,195,3600,maschio
Chinstrap,Dream,47.5,16.8,199,3900,femmina
Chinstrap,Dream,47.6,18.3,195,3850,femmina
Chinstrap,Dream,52,20.7,210,4800,maschio
Chinstrap,Dream,46.9,16.6,192,2700,femmina
Chinstrap,Dream,53.5,19.9,205,4500,maschio
Chinstrap,Dream,49,19.5,210,3950,maschio
Chinstrap,Dream,46.2,17.5,187,3650,femmina
Chinstrap,Dream,50.9,19.1,196,3550,maschio
Chinstrap,Dream,45.5,17,196,3500,femmina
Chinstrap,Dream,50.9,17.9,196,3675,femmina
Chinstrap,Dream,50.8,18.5,201,4450,maschio
Chinstrap,Dream,50.1,17.9,190,3400,femmina
Chinstrap,Dream,49,19.6,212,4300,maschio
Chinstrap,Dream,51.5,18.7,187,3250,maschio
Chinstrap,Dream,49.8,17.3,198,3675,femmina
Chinstrap,Dream,48.1,16.4,199,3325,femmina
Chinstrap,Dream,51.4,19,201,3950,maschio
Chinstrap,Dream,45.7,17.3,193,3600,femmina
Chinstrap,Dream,50.7,19.7,203,4050,maschio
Chinstrap,Dream,42.5,17.3,187,3350,femmina
Chinstrap,Dream,52.2,18.8,197,3450,maschio
Chinstrap,Dream,45.2,16.6,191,3250,femmina
Chinstrap,Dream,49.3,19.9,203,4050,maschio
Chinstrap,Dream,50.2,18.8,202,3800,maschio
Chinstrap,Dream,45.6,19.4,194,3525,femmina
Chinstrap,Dream,51.9,19.5,206,3950,maschio
Chinstrap,Dream,46.8,16.5,189,3650,femmina
Chinstrap,Dream,45.7,17,195,3650,femmina
Chinstrap,Dream,55.8,19.8,207,4000,maschio
Chinstrap,Dream,43.5,18.1,202,3400,femmina
Chinstrap,Dream,49.6,18.2,193,3775,maschio
Chinstrap,Dream,50.8,19,210,4100,maschio
Chinstrap,Dream,50.2,18.7,198,3775,femmina
Gentoo,Biscoe,46.1,13.2,211,4500,femmina
Gentoo,Biscoe,50,16.3,230,5700,maschio
Gentoo,Biscoe,48.7,14.1,210,4450,femmina
Gentoo,Biscoe,50,15.2,218,5700,maschio
Gentoo,Biscoe,47.6,14.5,215,5400,maschio
Gentoo,Biscoe,46.5,13.5,210,4550,femmina
Gentoo,Biscoe,45.4,14.6,211,4800,femmina
Gentoo,Biscoe,46.7,15.3,219,5200,maschio
Gentoo,Biscoe,43.3,13.4,209,4400,femmina
Gentoo,Biscoe,46.8,15.4,215,5150,maschio
Gentoo,Biscoe,40.9,13.7,214,4650,femmina
Gentoo,Biscoe,49,16.1,216,5550,maschio
Gentoo,Biscoe,45.5,13.7,214,4650,femmina
Gentoo,Biscoe,48.4,14.6,213,5850,maschio
Gentoo,Biscoe,45.8,14.6,210,4200,femmina
Gentoo,Biscoe,49.3,15.7,217,5850,maschio
Gentoo,Biscoe,42,13.5,210,4150,femmina
Gentoo,Biscoe,49.2,15.2,221,6300,maschio
Gentoo,Biscoe,46.2,14.5,209,4800,femmina
Gentoo,Biscoe,48.7,15.1,222,5350,maschio
Gentoo,Biscoe,50.2,14.3,218,5700,maschio
Gentoo,Biscoe,45.1,14.5,215,5000,femmina
Gentoo,Biscoe,46.5,14.5,213,4400,femmina
Gentoo,Biscoe,46.3,15.8,215,5050,maschio
Gentoo,Biscoe,42.9,13.1,215,5000,femmina
Gentoo,Biscoe,46.1,15.1,215,5100,maschio
Gentoo,Biscoe,44.5,14.3,216,4100,
Gentoo,Biscoe,47.8,15,215,5650,maschio
Gentoo,Biscoe,48.2,14.3,210,4600,femmina
Gentoo,Biscoe,50,15.3,220,5550,maschio
Gentoo,Biscoe,47.3,15.3,222,5250,maschio
Gentoo,Biscoe,42.8,14.2,209,4700,femmina
Gentoo,Biscoe,45.1,14.5,207,5050,femmina
Gentoo,Biscoe,59.6,17,230,6050,maschio
Gentoo,Biscoe,49.1,14.8,220,5150,femmina
Gentoo,Biscoe,48.4,16.3,220,5400,maschio
Gentoo,Biscoe,42.6,13.7,213,4950,femmina
Gentoo,Biscoe,44.4,17.3,219,5250,maschio
Gentoo,Biscoe,44,13.6,208,4350,femmina
Gentoo,Biscoe,48.7,15.7,208,5350,maschio
Gentoo,Biscoe,42.7,13.7,208,3950,femmina
Gentoo,Biscoe,49.6,16,225,5700,maschio
Gentoo,Biscoe,45.3,13.7,210,4300,femmina
Gentoo,Biscoe,49.6,15,216,4750,maschio
Gentoo,Biscoe,50.5,15.9,222,5550,maschio
Gentoo,Biscoe,43.6,13.9,217,4900,femmina
Gentoo,Biscoe,45.5,13.9,210,4200,femmina
Gentoo,Biscoe,50.5,15.9,225,5400,maschio
Gentoo,Biscoe,44.9,13.3,213,5100,femmina
Gentoo,Biscoe,45.2,15.8,215,5300,maschio
Gentoo,Biscoe,46.6,14.2,210,4850,femmina
Gentoo,Biscoe,48.5,14.1,220,5300,maschio
Gentoo,Biscoe,45.1,14.4,210,4400,femmina
Gentoo,Biscoe,50.1,15,225,5000,maschio
Gentoo,Biscoe,46.5,14.4,217,4900,femmina
Gentoo,Biscoe,45,15.4,220,5050,maschio
Gentoo,Biscoe,43.8,13.9,208,4300,femmina
Gentoo,Biscoe,45.5,15,220,5000,maschio
Gentoo,Biscoe,43.2,14.5,208,4450,femmina
Gentoo,Biscoe,50.4,15.3,224,5550,maschio
Gentoo,Biscoe,45.3,13.8,208,4200,femmina
Gentoo,Biscoe,46.2,14.9,221,5300,maschio
Gentoo,Biscoe,45.7,13.9,214,4400,femmina
Gentoo,Biscoe,54.3,15.7,231,5650,maschio
Gentoo,Biscoe,45.8,14.2,219,4700,femmina
Gentoo,Biscoe,49.8,16.8,230,5700,maschio
Gentoo,Biscoe,46.2,14.4,214,4650,
Gentoo,Biscoe,49.5,16.2,229,5800,maschio
Gentoo,Biscoe,43.5,14.2,220,4700,femmina
Gentoo,Biscoe,50.7,15,223,5550,maschio
Gentoo,Biscoe,47.7,15,216,4750,femmina
Gentoo,Biscoe,46.4,15.6,221,5000,maschio
Gentoo,Biscoe,48.2,15.6,221,5100,maschio
Gentoo,Biscoe,46.5,14.8,217,5200,femmina
Gentoo,Biscoe,46.4,15,216,4700,femmina
Gentoo,Biscoe,48.6,16,230,5800,maschio
Gentoo,Biscoe,47.5,14.2,209,4600,femmina
Gentoo,Biscoe,51.1,16.3,220,6000,maschio
Gentoo,Biscoe,45.2,13.8,215,4750,femmina
Gentoo,Biscoe,45.2,16.4,223,5950,maschio
Gentoo,Biscoe,49.1,14.5,212,4625,femmina
Gentoo,Biscoe,52.5,15.6,221,5450,maschio
Gentoo,Biscoe,47.4,14.6,212,4725,femmina
Gentoo,Biscoe,50,15.9,224,5350,maschio
Gentoo,Biscoe,44.9,13.8,212,4750,femmina
Gentoo,Biscoe,50.8,17.3,228,5600,maschio
Gentoo,Biscoe,43.4,14.4,218,4600,femmina
Gentoo,Biscoe,51.3,14.2,218,5300,maschio
Gentoo,Biscoe,47.5,14,212,4875,femmina
Gentoo,Biscoe,52.1,17,230,5550,maschio
Gentoo,Biscoe,47.5,15,218,4950,femmina
Gentoo,Biscoe,52.2,17.1,228,5400,maschio
Gentoo,Biscoe,45.5,14.5,212,4750,femmina
Gentoo,Biscoe,49.5,16.1,224,5650,maschio
Gentoo,Biscoe,44.5,14.7,214,4850,femmina
Gentoo,Biscoe,50.8,15.7,226,5200,maschio
Gentoo,Biscoe,49.4,15.8,216,4925,maschio
Gentoo,Biscoe,46.9,14.6,222,4875,femmina
Gentoo,Biscoe,48.4,14.4,203,4625,femmina
Gentoo,Biscoe,51.1,16.5,225,5250,maschio
Gentoo,Biscoe,48.5,15,219,4850,femmina
Gentoo,Biscoe,55.9,17,228,5600,maschio
Gentoo,Biscoe,47.2,15.5,215,4975,femmina
Gentoo,Biscoe,49.1,15,228,5500,maschio
Gentoo,Biscoe,47.3,13.8,216,4725,
Gentoo,Biscoe,46.8,16.1,215,5500,maschio
Gentoo,Biscoe,41.7,14.7,210,4700,femmina
Gentoo,Biscoe,53.4,15.8,219,5500,maschio
Gentoo,Biscoe,43.3,14,208,4575,femmina
Gentoo,Biscoe,48.1,15.1,209,5500,maschio
Gentoo,Biscoe,50.5,15.2,216,5000,femmina
Gentoo,Biscoe,49.8,15.9,229,5950,maschio
Gentoo,Biscoe,43.5,15.2,213,4650,femmina
Gentoo,Biscoe,51.5,16.3,230,5500,maschio
Gentoo,Biscoe,46.2,14.1,217,4375,femmina
Gentoo,Biscoe,55.1,16,230,5850,maschio
Gentoo,Biscoe,44.5,15.7,217,4875,
Gentoo,Biscoe,48.8,16.2,222,6000,maschio
Gentoo,Biscoe,47.2,13.7,214,4925,femmina
Gentoo,Biscoe,,,,,
Gentoo,Biscoe,46.8,14.3,215,4850,femmina
Gentoo,Biscoe,50.4,15.7,222,5750,maschio
Gentoo,Biscoe,45.2,14.8,212,5200,femmina
Gentoo,Biscoe,49.9,16.1,213,5400,maschio
"""

with open("pinguini.csv", "w", encoding="utf-8") as f:
    f.write(PINGUINI_CSV)

print("✅ File pinguini.csv creato!")

In [ ]:
#@title 🛠️ Funzioni di supporto per i grafici (esegui la cella, non serve leggerla)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Un colore fisso per ogni classe, così lo stesso pinguino ha sempre lo stesso colore
COLORI = {"Adelie": "#FF8C00", "Chinstrap": "#A034F0", "Gentoo": "#159090",
          "femmina": "#D9534F", "maschio": "#2C7FB8"}
# I cluster non hanno un nome: colori diversi da quelli delle specie, per non confonderli
COLORI_CLUSTER = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3"]

def predici(modello, X):
    """Predizioni per X: funziona sia con un modello sklearn (che ha .predict)
    sia con una funzione scritta da noi che riceve un DataFrame."""
    if hasattr(modello, "predict"):
        return np.asarray(modello.predict(X))
    return np.asarray(modello(X))

def _griglia(X, passi=300):
    c1, c2 = X.columns
    m1 = (X[c1].max() - X[c1].min()) * 0.05
    m2 = (X[c2].max() - X[c2].min()) * 0.05
    xx, yy = np.meshgrid(np.linspace(X[c1].min() - m1, X[c1].max() + m1, passi),
                         np.linspace(X[c2].min() - m2, X[c2].max() + m2, passi))
    return xx, yy, pd.DataFrame({c1: xx.ravel(), c2: yy.ravel()})

def mostra_punti(X, y, titolo="", ax=None, dimensioni=35):
    """Scatter plot di due feature, un colore per classe.
    `dimensioni`: un numero (uguale per tutti) oppure una dimensione per ogni punto."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    c1, c2 = X.columns
    for classe in sorted(pd.unique(np.asarray(y))):
        sel = np.asarray(y) == classe
        s = dimensioni if np.isscalar(dimensioni) else np.asarray(dimensioni)[sel]
        ax.scatter(X[c1][sel], X[c2][sel], c=COLORI.get(classe, "gray"), label=classe,
                   edgecolors="k", linewidths=0.5, s=s)
    ax.set_xlabel(c1); ax.set_ylabel(c2); ax.set_title(titolo)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    if nuovo:
        plt.show()

def disegna_frontiera(modello, X, y, titolo="", ax=None, dimensioni=35):
    """Colora il piano secondo la classe predetta dal modello (frontiera decisionale)."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    xx, yy, griglia = _griglia(X)
    pred = predici(modello, griglia)
    classi = sorted(set(pd.unique(pred)) | set(pd.unique(np.asarray(y))))
    indice = {c: i for i, c in enumerate(classi)}
    z = np.array([indice[p] for p in pred]).reshape(xx.shape)
    cmap = ListedColormap([COLORI.get(c, "gray") for c in classi])
    ax.contourf(xx, yy, z, alpha=0.25, cmap=cmap, levels=np.arange(len(classi) + 1) - 0.5)
    mostra_punti(X, y, titolo, ax=ax, dimensioni=dimensioni)
    if nuovo:
        plt.show()

def mostra_vicini(X, y, nuovo, k, ax=None):
    """Zoom attorno al punto `nuovo` (una tabella con una riga sola):
    la stella è il punto nuovo, i cerchi neri sono i suoi k vicini."""
    nuovo_grafico = ax is None
    if nuovo_grafico:
        fig, ax = plt.subplots(figsize=(6.5, 6))
    c1, c2 = X.columns
    p = nuovo.iloc[0]
    distanze = ((X[c1] - p[c1]) ** 2 + (X[c2] - p[c2]) ** 2) ** 0.5
    vicini = distanze.sort_values().index[:k]
    raggio = distanze.loc[vicini].max()
    mostra_punti(X, y, ax=ax, dimensioni=60)
    ax.scatter(X.loc[vicini, c1], X.loc[vicini, c2], s=330, facecolors="none", edgecolors="k",
               linewidths=2, label=f"i {k} più vicini")
    ax.add_patch(plt.Circle((p[c1], p[c2]), raggio, fill=False, linestyle="--", color="k"))
    ax.scatter([p[c1]], [p[c2]], marker="*", s=450, c="gold", edgecolors="k", zorder=5, label="pinguino nuovo")
    zoom = max(2.2 * raggio, 2.5)
    ax.set_xlim(p[c1] - zoom, p[c1] + zoom); ax.set_ylim(p[c2] - zoom, p[c2] + zoom)
    ax.set_aspect("equal")            # stessa scala sui due assi: il cerchio tratteggiato è un vero cerchio
    ax.set_title(f"I {k} pinguini più vicini al pinguino nuovo (zoom)")
    ax.legend(fontsize=8, loc="upper left", markerscale=0.45)
    if nuovo_grafico:
        plt.show()

def mostra_cluster(X, etichette, centroidi=None, titolo="", ax=None):
    """Scatter plot di due feature colorato per cluster; le ✕ sono i centroidi."""
    nuovo = ax is None
    if nuovo:
        fig, ax = plt.subplots(figsize=(7, 5))
    c1, c2 = X.columns
    etichette = np.asarray(etichette)
    for j in sorted(np.unique(etichette)):
        sel = etichette == j
        ax.scatter(X[c1][sel], X[c2][sel], c=COLORI_CLUSTER[j % len(COLORI_CLUSTER)], label=f"cluster {j}",
                   edgecolors="k", linewidths=0.5, s=35)
    if centroidi is not None:
        centroidi = np.asarray(centroidi)
        ax.scatter(centroidi[:, 0], centroidi[:, 1], marker="X", s=260, c="k", edgecolors="w", linewidths=1.5,
                   zorder=5, label="centroidi")
    ax.set_xlabel(c1); ax.set_ylabel(c2); ax.set_title(titolo)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    if nuovo:
        plt.show()

def passi_kmeans(Z, centroidi_iniziali, max_passi=40):
    """Esegue k-means un passo alla volta e restituisce la lista degli stati attraversati.
    Ogni stato è un dizionario: tipo del passo, etichette, centroidi, centroidi precedenti, inertia."""
    Z = np.asarray(Z, dtype=float)
    centroidi = np.asarray(centroidi_iniziali, dtype=float)
    stati = [{"tipo": "inizio", "punti": Z, "etichette": None, "centroidi": centroidi,
              "precedenti": None, "inertia": None}]
    etichette = None
    while len(stati) <= max_passi:
        # assegnazione: ogni punto va al centroide più vicino
        distanze2 = ((Z[:, None, :] - centroidi[None, :, :]) ** 2).sum(axis=2)
        nuove = distanze2.argmin(axis=1)
        if etichette is not None and (nuove == etichette).all():
            break                                       # nessun punto ha cambiato cluster: convergenza
        etichette = nuove
        stati.append({"tipo": "assegnazione", "punti": Z, "etichette": etichette, "centroidi": centroidi,
                      "precedenti": None, "inertia": distanze2.min(axis=1).sum()})
        # aggiornamento: ogni centroide si sposta nella media dei suoi punti
        vecchi = centroidi
        centroidi = np.array([Z[etichette == j].mean(axis=0) if (etichette == j).any() else vecchi[j]
                              for j in range(len(vecchi))])
        stati.append({"tipo": "aggiornamento", "punti": Z, "etichette": etichette, "centroidi": centroidi,
                      "precedenti": vecchi, "inertia": ((Z - centroidi[etichette]) ** 2).sum()})
    return stati

def disegna_passo_kmeans(stato, ax, passo=None, nomi=("feature 1", "feature 2")):
    """Disegna uno degli stati restituiti da passi_kmeans."""
    Z, centroidi, etichette = stato["punti"], stato["centroidi"], stato["etichette"]
    ax.clear()
    if etichette is None:
        ax.scatter(Z[:, 0], Z[:, 1], c="lightgray", edgecolors="k", linewidths=0.4, s=30)
    else:
        colori = [COLORI_CLUSTER[j % len(COLORI_CLUSTER)] for j in etichette]
        ax.scatter(Z[:, 0], Z[:, 1], c=colori, edgecolors="k", linewidths=0.4, s=30)
    if stato["precedenti"] is not None:                 # frecce dal vecchio al nuovo centroide
        for vecchio, nuovo in zip(stato["precedenti"], centroidi):
            ax.annotate("", xy=nuovo, xytext=vecchio, arrowprops={"arrowstyle": "->", "color": "k", "lw": 1.5, "shrinkB": 9})
    for j, c in enumerate(centroidi):
        ax.scatter([c[0]], [c[1]], marker="X", s=280, c=COLORI_CLUSTER[j % len(COLORI_CLUSTER)],
                   edgecolors="k", linewidths=1.5, zorder=5)
    descrizione = {"inizio": "centroidi iniziali",
                   "assegnazione": "assegnazione (ogni punto al centroide più vicino)",
                   "aggiornamento": "aggiornamento (ogni centroide nella media dei suoi punti)"}[stato["tipo"]]
    titolo = descrizione if passo is None else f"passo {passo}: {descrizione}"
    if stato["inertia"] is not None:
        titolo += f"\ninertia = {stato['inertia']:.1f}"
    ax.set_title(titolo, fontsize=10)
    ax.set_xlabel(nomi[0]); ax.set_ylabel(nomi[1]); ax.grid(alpha=0.3)
    ax.set_xlim(Z[:, 0].min() - 0.3, Z[:, 0].max() + 0.3)     # limiti fissi: il grafico non "salta" tra un passo e l'altro
    ax.set_ylim(Z[:, 1].min() - 0.3, Z[:, 1].max() + 0.3)

print("✅ Funzioni pronte: mostra_punti, disegna_frontiera, mostra_vicini, mostra_cluster, passi_kmeans, disegna_passo_kmeans")

### 🔁 Dall'Esercitazione 2 ci portiamo dietro
- Ogni modello di scikit-learn si usa allo stesso modo: lo **creo** scegliendo gli iperparametri, lo **addestro** con `.fit(X, y)`, lo **uso** con `.predict(X)`.
- Per **scegliere** un modello non si guarda il test set: si usa la **cross-validation** sul training set. `StratifiedKFold` divide il training set in 5 fold, `cross_val_score` restituisce i 5 punteggi di validazione, di cui guardiamo **media** e **deviazione standard**.
- `GridSearchCV` prova tutte le combinazioni di iperparametri di una griglia, valuta ognuna con la cross-validation e tiene la migliore.
- 📌 Una differenza tra due punteggi di cross-validation conta solo se è più grande della loro deviazione standard. Quando due modelli sono alla pari, conviene il più semplice.
- Per stimare il **sesso** di un pinguino dalle due misure del becco avevamo scelto un albero con `max_depth=5` e `min_samples_leaf=10`: accuracy in cross-validation **0.82**.

---
# Parte 1 — Ensemble: tanti alberi invece di uno

Ripartiamo dallo stesso problema dell'Esercitazione 2, con gli stessi dati e la stessa divisione in training e test set.

> 📋 **Il problema: è maschio o femmina?**
> - **Dati:** i 333 pinguini senza valori mancanti
> - **Target `y`:** `sesso`: maschio o femmina
> - **Feature `X`:** `becco_lunghezza` e `becco_profondita` (con due feature possiamo disegnare la frontiera)
> - **Tipo di problema:** **classificazione binaria**
> - **Come lo valutiamo:** accuracy, stimata con la cross-validation sul training set

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

pinguini = pd.read_csv("pinguini.csv")
df = pinguini.dropna()          # elimina le righe con valori mancanti: restano 333 pinguini

X = df[["becco_lunghezza", "becco_profondita"]]
y = df["sesso"]

# stratify=y: training e test con la stessa proporzione di maschi e femmine
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# la stessa cross-validation dell'Esercitazione 2: 5 fold stratificati
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print("Pinguini:", len(df), "| training:", len(X_train), "| test:", len(X_test))

🔒 **Il test set resta chiuso** fino alla 2.5, dove sceglieremo il modello finale.

### 1.1 Il punto di partenza
Rifacciamo i due alberi dell'Esercitazione 2: quello **senza limiti** e quello **scelto** con `GridSearchCV`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

albero_libero = DecisionTreeClassifier(random_state=0)                                     # senza limiti
albero_scelto = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=0)   # scelto nell'Esercitazione 2

punteggi_libero = cross_val_score(albero_libero, X_train, y_train, cv=cv)    # i 5 punteggi di validazione
punteggi_scelto = cross_val_score(albero_scelto, X_train, y_train, cv=cv)
print(f"Albero senza limiti – accuracy in CV: {punteggi_libero.mean():.2f} ± {punteggi_libero.std():.2f}")
print(f"Albero scelto       – accuracy in CV: {punteggi_scelto.mean():.2f} ± {punteggi_scelto.std():.2f}")

albero_libero.fit(X_train, y_train)      # cross_val_score non addestra il modello che gli passiamo: lo facciamo qui
albero_scelto.fit(X_train, y_train)
fig, assi = plt.subplots(1, 2, figsize=(15, 5))
disegna_frontiera(albero_libero, X_train, y_train, "Albero senza limiti", ax=assi[0])
disegna_frontiera(albero_scelto, X_train, y_train, "Albero scelto: max_depth=5, min_samples_leaf=10", ax=assi[1])
plt.show()

L'albero senza limiti costruisce **isolette** attorno a singoli pinguini e in cross-validation si ferma a 0.78; quello scelto ha una frontiera più semplice e arriva a 0.82. Da qui in avanti 0.82 è il numero da battere.

### 1.2 Un albero varia molto
Quanto cambia un albero se i dati cambiano di poco? Creiamo 4 **campioni bootstrap** del training set: estrazioni **con reinserimento** della stessa dimensione dell'originale (alcuni pinguini compaiono più volte, altri mai). Su ognuno addestriamo un albero senza limiti. In ogni riquadro vedi solo i pinguini di quel campione: più un punto è grande, più volte quel pinguino è stato estratto; quelli mai estratti non compaiono.

> 🐍 `rng = np.random.default_rng(0)` crea un generatore di numeri casuali con seme fisso. `rng.choice(n, size=n, replace=True)` estrae `n` numeri tra 0 e `n-1` **con reinserimento**. `.iloc[indici]` prende le righe in quelle posizioni. `enumerate(...)` dà anche il numero del giro: 0, 1, 2, 3. `np.bincount(indici, minlength=n)` conta quante volte compare ogni numero da 0 a `n-1`.

In [ ]:
rng = np.random.default_rng(0)
n = len(X_train)

fig, assi = plt.subplots(2, 2, figsize=(14, 10))
for i, ax in enumerate(assi.ravel()):                               # .ravel(): i 4 riquadri uno dopo l'altro
    indici = rng.choice(n, size=n, replace=True)                    # un campione bootstrap
    X_boot, y_boot = X_train.iloc[indici], y_train.iloc[indici]
    albero_boot = DecisionTreeClassifier(random_state=0).fit(X_boot, y_boot)
    volte = np.bincount(indici, minlength=n)                        # quante volte è stato estratto ogni pinguino
    # punto più grande = pinguino estratto più volte; i pinguini mai estratti (0 volte) non si vedono
    disegna_frontiera(albero_boot, X_train, y_train, f"Albero {i + 1} (campione bootstrap {i + 1})",
                      ax=ax, dimensioni=25 * volte)
plt.tight_layout()
plt.show()

Le quattro frontiere sono molto diverse, anche se i quattro campioni vengono dagli stessi 233 pinguini e si somigliano molto.

📌 **Bias e varianza.** Immagina un bersaglio: il centro è la risposta giusta, e ogni freccia è un modello addestrato su un campione diverso dei dati.
- **Alta varianza**: le frecce sono sparpagliate. Il modello cambia molto da un campione all'altro, perché insegue i singoli punti. È il caso degli alberi **profondi**, come questi quattro.
- **Alto bias**: le frecce sono vicine tra loro, ma tutte lontane dal centro. Il modello è troppo semplice e sbaglia sempre nello stesso modo. È il caso degli alberi **bassi**, per esempio con `max_depth=1`.

Gli ensemble mettono insieme tanti modelli per ridurre uno dei due problemi.

### 1.3 Il bootstrap da vicino
Quanti pinguini **diversi** finiscono in un campione bootstrap? Ripetiamo l'estrazione 1000 volte.

In [ ]:
# per 1000 volte: estraggo un campione bootstrap e calcolo la frazione di pinguini diversi
frazioni = [len(np.unique(rng.choice(n, size=n, replace=True))) / n for _ in range(1000)]   # _ = variabile che non usiamo
print(f"Pinguini diversi in un campione, in media: {np.mean(frazioni):.1%}")

In ogni campione c'è circa il **63%** dei pinguini. Il restante **37%**, mai visto da quell'albero, sono i suoi esempi **out-of-bag** (OOB): si possono usare per valutarlo, come un piccolo validation set che non costa nulla.

### 1.4 Random forest
Una **random forest** addestra tanti alberi profondi, ognuno su un campione bootstrap diverso, e li fa **votare**: vince la classe scelta dalla maggioranza. Gli errori dei singoli alberi sono diversi tra loro e nel voto in parte si compensano: è il **bagging** (*bootstrap aggregating*), e riduce la **varianza**. In più, a ogni nodo ogni albero può scegliere solo tra **alcune feature estratte a caso** (`max_features`), così gli alberi si somigliano ancora meno.

Con `oob_score=True` la foresta calcola anche la stima out-of-bag: ogni pinguino viene classificato solo dagli alberi che non l'hanno visto.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

foresta = RandomForestClassifier(n_estimators=200,      # 200 alberi
                                 oob_score=True,        # calcola la stima out-of-bag
                                 random_state=0)
foresta.fit(X_train, y_train)

punteggi_foresta = cross_val_score(foresta, X_train, y_train, cv=cv)
print(f"Random forest – stima out-of-bag (oob_score_): {foresta.oob_score_:.2f}")
print(f"Random forest – accuracy in CV:                {punteggi_foresta.mean():.2f} ± {punteggi_foresta.std():.2f}")
print(f"Albero senza limiti – accuracy in CV:          {punteggi_libero.mean():.2f} ± {punteggi_libero.std():.2f}")
print(f"Albero scelto – accuracy in CV:                {punteggi_scelto.mean():.2f} ± {punteggi_scelto.std():.2f}")

fig, assi = plt.subplots(1, 2, figsize=(15, 5))
disegna_frontiera(albero_libero, X_train, y_train, "Un albero senza limiti", ax=assi[0])
disegna_frontiera(foresta, X_train, y_train, "Random forest: 200 alberi senza limiti, a voto", ax=assi[1])
plt.show()

📌 La stima out-of-bag (0.81) è vicina alla cross-validation (0.80), con un solo addestramento invece di cinque. Un albero senza limiti da solo arriva a 0.78; duecento alberi senza limiti che votano, senza nessuna messa a punto, arrivano a 0.80. È un miglioramento più piccolo della deviazione standard, e l'albero scelto (0.82) non viene battuto: **con due sole feature la foresta non aggiunge niente**, e 200 alberi non si leggono come un `if`/`else`.

Quanti alberi servono? Calcoliamo l'accuracy in cross-validation con un numero crescente di alberi (`n_estimators`):

In [ ]:
numero_alberi = [1, 2, 5, 10, 20, 50, 100, 200]
acc_foresta = []

for b in numero_alberi:
    foresta_b = RandomForestClassifier(n_estimators=b, random_state=0)
    acc_foresta.append(cross_val_score(foresta_b, X_train, y_train, cv=cv).mean())

print("Accuracy in CV:", dict(zip(numero_alberi, np.round(acc_foresta, 2).tolist())))   # zip accoppia le due liste
plt.figure(figsize=(8, 4.5))
plt.plot(numero_alberi, acc_foresta, marker="o")
plt.xscale("log"); plt.xticks(numero_alberi, numero_alberi); plt.xlim(0.8, 250)
plt.xlabel("numero di alberi (scala log)"); plt.ylabel("accuracy in cross-validation"); plt.grid(alpha=0.3)
plt.title("Accuracy della foresta al variare del numero di alberi")
plt.show()

Con 1 o 2 alberi l'accuracy è 0.76, come un albero solo; con 5 sale a 0.81, poi resta tra 0.79 e 0.81: le oscillazioni sono rumore della stima. Aggiungere alberi non fa peggiorare la foresta, ma dopo poche decine non la migliora più e costa solo tempo.

### 1.5 Boosting
Il **boosting** usa gli alberi in un altro modo: li addestra **in sequenza**, e ogni albero corregge gli errori di quelli che lo precedono. Gli alberi sono **poco profondi** (in `GradientBoostingClassifier` di default `max_depth=3`): ognuno da solo è troppo semplice, ma sommandone tanti il modello diventa via via più preciso.

Due iperparametri vanno scelti insieme: `n_estimators` (quanti alberi) e `learning_rate` (quanto pesa la correzione di ogni albero). Li cerchiamo con `GridSearchCV`, come nell'Esercitazione 2; `n_jobs=-1` usa tutti i processori, per fare prima.

> 🐍 `.pivot_table(index=..., columns=..., values=...)` riorganizza una tabella: una riga per ogni valore di `index`, una colonna per ogni valore di `columns`, e nelle caselle i valori di `values`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

griglia_boosting = {
    "learning_rate": [0.01, 0.1, 0.5],
    "n_estimators": [50, 100, 200, 500],
}
ricerca_boosting = GridSearchCV(GradientBoostingClassifier(random_state=0), griglia_boosting, cv=cv, n_jobs=-1)
ricerca_boosting.fit(X_train, y_train)

print("Migliore:", ricerca_boosting.best_params_, f"→ accuracy in CV {ricerca_boosting.best_score_:.2f}")
# l'accuracy media in CV di ogni combinazione: righe = learning_rate, colonne = n_estimators
pd.DataFrame(ricerca_boosting.cv_results_).pivot_table(
    index="param_learning_rate", columns="param_n_estimators", values="mean_test_score").round(2)

Tutte le combinazioni stanno tra 0.78 e 0.83, e la migliore (0.83) è alla pari con l'albero scelto (0.82). Guarda la riga `learning_rate = 0.1`: passando da 50 a 500 alberi l'accuracy **scende**, da 0.82 a 0.78. La differenza è piccola, ma va nella direzione opposta a quella della foresta: nel boosting ogni albero in più insegue gli errori rimasti, e oltre un certo punto insegue il rumore del training set (**overfitting**).

Invece di provare tanti valori di `n_estimators` si può usare l'**early stopping**: si mette da parte una frazione del training set (`validation_fraction`) e ci si ferma quando per `n_iter_no_change` alberi di fila il punteggio su quella parte non migliora.

In [ ]:
boosting_es = GradientBoostingClassifier(n_estimators=1000,         # al massimo 1000 alberi
                                         learning_rate=0.1,
                                         validation_fraction=0.2,   # il 20% del training per decidere quando fermarsi
                                         n_iter_no_change=10,       # stop dopo 10 alberi senza miglioramenti
                                         random_state=0)
boosting_es.fit(X_train, y_train)
print("Alberi usati davvero (n_estimators_):", boosting_es.n_estimators_, "su 1000")
print(f"Accuracy in CV: {cross_val_score(boosting_es, X_train, y_train, cv=cv).mean():.2f}")

Il boosting si ferma da solo dopo 39 alberi, con un'accuracy in cross-validation di 0.81: ancora alla pari con l'albero scelto. Con due sole feature nessun ensemble fa meglio di un albero ben regolato; nella Parte 2 li riproveremo con tutte le misure.

📌 **Bagging e boosting a confronto.**
- Il **bagging** (random forest) addestra alberi **profondi** e **indipendenti** tra loro, su campioni bootstrap, e li fa votare: riduce la **varianza**. Gli alberi si possono addestrare in parallelo, e aggiungerne altri non porta overfitting.
- Il **boosting** addestra alberi **bassi** uno **dopo l'altro**, ognuno sugli errori dei precedenti, e li somma: riduce il **bias**. Non si può parallelizzare allo stesso modo, e con troppi alberi può andare in overfitting: per questo servono `learning_rate` ed early stopping.

### 1.6 ⏩ AdaBoost passo per passo
**AdaBoost** è il boosting più semplice da seguire a mano. Ogni pinguino ha un **peso** $w_i$, all'inizio uguale per tutti ($1/n$). A ogni round:
1. si addestra un albero con una sola domanda (uno *stump*, `max_depth=1`) che tiene conto dei pesi (`sample_weight`);
2. si calcola il suo **errore pesato** $\varepsilon$: la somma dei pesi dei pinguini che sbaglia;
3. si calcola quanto conta il suo voto: $\alpha = \tfrac12 \ln\frac{1-\varepsilon}{\varepsilon}$ (più l'errore è basso, più $\alpha$ è grande);
4. si aggiornano i pesi: $w_i \leftarrow w_i \, e^{-\alpha \, y_i \, h(x_i)}$, poi si dividono per la loro somma, così tornano a sommare a 1.

Qui le classi sono scritte come numeri: $y_i = +1$ per i maschi e $-1$ per le femmine, e $h(x_i)$ è la predizione dello stump, sempre $+1$ o $-1$. Se la predizione è giusta $y_i \, h(x_i) = +1$ e il peso **scende**; se è sbagliata $y_i \, h(x_i) = -1$ e il peso **sale**: lo stump successivo si concentrerà sui pinguini sbagliati.

Nei grafici la **dimensione** di ogni punto è proporzionale al peso con cui quel pinguino entra nel round.

> 🐍 `np.where(condizione, a, b)` restituisce `a` dove la condizione è vera e `b` dove è falsa. `np.full(n, valore)` crea un array di `n` elementi tutti uguali a `valore`. `np.exp` e `np.log` sono l'esponenziale e il logaritmo naturale.

In [ ]:
y_num = np.where(y_train == "maschio", 1, -1)      # +1 = maschio, -1 = femmina
pesi = np.full(n, 1 / n)                           # all'inizio tutti i pinguini pesano 1/n
voto = np.zeros(n)                                 # la somma dei voti degli stump, per ogni pinguino

fig, assi = plt.subplots(1, 3, figsize=(18, 5))
for t, ax in enumerate(assi):
    stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train, sample_weight=pesi)
    h = np.where(stump.predict(X_train) == "maschio", 1, -1)     # predizioni dello stump: +1 o -1
    errore = pesi[h != y_num].sum()                              # ε: somma dei pesi dei pinguini sbagliati
    alpha = 0.5 * np.log((1 - errore) / errore)                  # quanto conta il voto di questo stump
    disegna_frontiera(stump, X_train, y_train, f"Round {t + 1}: ε = {errore:.3f}, α = {alpha:.3f}",
                      ax=ax, dimensioni=pesi * n * 30)           # punti grandi = pinguini che pesano di più
    voto = voto + alpha * h
    pesi = pesi * np.exp(-alpha * y_num * h)                     # giusti: peso giù; sbagliati: peso su
    pesi = pesi / pesi.sum()                                     # normalizzazione: la somma torna 1
    print(f"Round {t + 1}: ε = {errore:.3f}, α = {alpha:.3f} | accuracy dello stump {(h == y_num).mean():.2f}"
          f" | accuracy del voto dei primi {t + 1}: {(np.sign(voto) == y_num).mean():.2f}")
plt.tight_layout()
plt.show()

Il primo stump divide sulla lunghezza del becco e indovina il 71% dei pinguini. Nel secondo round i pinguini che ha sbagliato sono più grandi, e lo stump sceglie un'altra domanda, sulla profondità. Il terzo stump, preso da solo, indovina appena il 61% dei pinguini, eppure il **voto pesato** dei tre ($\text{segno}(\sum_t \alpha_t h_t(x))$) arriva a 0.84 sul training set: ognuno copre gli errori degli altri.

`AdaBoostClassifier` fa esattamente questo. Di default usa proprio stump come modelli di base:

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=3, random_state=0).fit(X_train, y_train)      # 3 round, come i nostri
ada_num = np.where(ada.predict(X_train) == "maschio", 1, -1)
print("Stesse predizioni del nostro calcolo a mano?", (ada_num == np.sign(voto)).all())

for numero in [3, 50]:
    punteggi_ada = cross_val_score(AdaBoostClassifier(n_estimators=numero, random_state=0), X_train, y_train, cv=cv)
    print(f"AdaBoost con {numero:2d} stump – accuracy in CV: {punteggi_ada.mean():.2f} ± {punteggi_ada.std():.2f}")

Con 3 stump l'accuracy in cross-validation è 0.79; con 50 sale a 0.82, di nuovo alla pari con l'albero scelto.

---
# Parte 2 — KNN: decidere guardando i vicini

> 📋 **Il problema: è maschio o femmina?** (lo stesso della Parte 1, con un modello diverso)
> - **Dati, target, feature:** come nella Parte 1; dalla 2.3 useremo tutte e quattro le misure
> - **Tipo di problema:** **classificazione binaria**
> - **Come lo valutiamo:** accuracy in cross-validation sul training set; il test set 🔒 si apre nella 2.5

### 2.1 KNN a mano
Il **KNN** (*k-nearest neighbors*, i $k$ vicini più vicini) non costruisce regole. Per classificare un pinguino nuovo:
1. calcola la **distanza** tra il pinguino nuovo e **tutti** i pinguini del training set;
2. prende i $k$ più vicini;
3. li fa **votare**: vince il sesso più frequente tra loro.

La distanza è quella **euclidea**, cioè il teorema di Pitagora: $d = \sqrt{(x_1 - x'_1)^2 + (x_2 - x'_2)^2}$.

Proviamo con un pinguino nuovo, con il becco lungo 45 mm e profondo 17 mm, e $k = 5$.

> 🐍 `** 2` eleva al quadrato e `** 0.5` fa la radice quadrata. Le operazioni tra una colonna e un numero si applicano a tutte le righe insieme, senza `for`. `.sort_values("distanza")` ordina la tabella dalla distanza più piccola, `.head(k)` tiene le prime `k` righe, `.value_counts()` conta quante volte compare ogni valore.

In [ ]:
lunghezza_nuovo, profondita_nuovo = 45.0, 17.0      # le misure del pinguino nuovo
k = 5

vicini = X_train.copy()                 # .copy(): una tabella nuova, così non modifichiamo X_train
vicini["sesso"] = y_train
vicini["distanza"] = ((vicini["becco_lunghezza"] - lunghezza_nuovo) ** 2
                      + (vicini["becco_profondita"] - profondita_nuovo) ** 2) ** 0.5   # Pitagora, riga per riga

piu_vicini = vicini.sort_values("distanza").head(k)     # i k pinguini con la distanza più piccola
print(piu_vicini["sesso"].value_counts())               # il voto
piu_vicini

Tra i 5 pinguini più vicini ci sono 4 femmine e 1 maschio: il KNN risponde **femmina**. Vediamoli nel grafico, e controlliamo che `KNeighborsClassifier` di scikit-learn dia la stessa risposta.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# il pinguino nuovo come tabella con una riga sola e le stesse colonne di X_train
nuovo = pd.DataFrame({"becco_lunghezza": [lunghezza_nuovo], "becco_profondita": [profondita_nuovo]})
mostra_vicini(X_train, y_train, nuovo, k)

knn = KNeighborsClassifier(n_neighbors=k)     # 1) creo il modello: k è un iperparametro
knn.fit(X_train, y_train)                     # 2) "addestramento": memorizza il training set
print("Risposta di KNeighborsClassifier:", knn.predict(nuovo))    # 3) qui avviene tutto il calcolo

📌 Il KNN **non ha un vero addestramento**: `fit` si limita a memorizzare il training set, e tutto il lavoro (distanze, ordinamento, voto) avviene in `predict`. Per questo si dice che è un algoritmo *lazy* (pigro). È il contrario dell'albero, che fatica in `fit` per costruire le regole e poi risponde subito.

### 2.2 Quanti vicini? Lo slider su k
$k$ è un **iperparametro**: lo scegliamo noi. Usa lo slider per cambiarlo (solo valori dispari, così tra due classi il voto non finisce mai in parità). Nel titolo vedi l'accuracy sul training set e quella media in cross-validation. Il grafico si aggiorna quando rilasci lo slider.

In [ ]:
from sklearn.metrics import accuracy_score
from ipywidgets import IntSlider, Image, VBox, Layout    # ipywidgets: elementi interattivi nei notebook
from IPython.display import display
import io                                                # per salvare il grafico in memoria invece che su file

fig_knn, ax_knn = plt.subplots(figsize=(6.5, 4.6))       # una sola figura, che ridisegniamo a ogni movimento
plt.close(fig_knn)                                       # non mostrarla qui: la mostriamo dentro `immagine_knn`

def prova_knn(k):
    """Addestra un KNN con k vicini e disegna la sua frontiera, con accuracy di training e in CV."""
    knn_k = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    acc_train = accuracy_score(y_train, knn_k.predict(X_train))
    acc_cv = cross_val_score(knn_k, X_train, y_train, cv=cv).mean()
    ax_knn.clear()                                       # cancella il disegno precedente
    disegna_frontiera(knn_k, X_train, y_train,
                      f"k = {k}\naccuracy training {acc_train:.2f} · accuracy CV {acc_cv:.2f}", ax=ax_knn)
    return fig_knn

# step=2: solo valori dispari; continuous_update=False: il grafico si aggiorna quando rilasci lo slider
slider_k = IntSlider(min=1, max=51, step=2, value=1, description="k",
                     continuous_update=False, layout=Layout(width="600px"))
immagine_knn = Image(format="png", layout=Layout(width="600px"))   # un'immagine che sostituiamo a ogni movimento

def aggiorna_knn(cambiamento=None):
    memoria = io.BytesIO()                               # un "file" che sta in memoria
    prova_knn(slider_k.value).savefig(memoria, format="png", bbox_inches="tight")
    immagine_knn.value = memoria.getvalue()              # al posto della vecchia immagine mette la nuova

slider_k.observe(aggiorna_knn, names="value")            # quando lo slider cambia, richiama aggiorna_knn
display(VBox([slider_k, immagine_knn]))
aggiorna_knn()

*Se lo slider non compare, puoi chiamare la funzione a mano, per esempio `prova_knn(5)`.*

> 🤔 Come cambia la frontiera passando da $k = 1$ a $k = 51$? Cosa succede all'accuracy di training, e a quella in cross-validation?

<details>
<summary><b>💡 Risposta</b> (clicca per aprire, ma prima prova a rispondere!)</summary>

Con $k = 1$ ogni pinguino del training set è il vicino più vicino di se stesso: l'accuracy di training è 1.00 e la frontiera è piena di isolette, ma in cross-validation l'accuracy è solo 0.76. È **overfitting**, cioè **varianza alta**: lo stesso comportamento dell'albero senza limiti.

Con $k = 3$ e $k = 5$ la frontiera si semplifica e la cross-validation sale a 0.82, come l'albero scelto.

Con $k = 51$ ogni risposta è il voto di 51 pinguini, più di un quinto del training set: la frontiera perde i dettagli, il training scende a 0.78 e la cross-validation a 0.77. Il modello è troppo rigido: **bias alto** (underfitting).

$k$ fa per il KNN quello che `max_depth` fa per l'albero, ma al contrario: $k$ **piccolo** vuol dire modello **complesso**.

</details>

### 2.3 La scala delle misure
Finora abbiamo usato solo il becco, per poter disegnare. Ora usiamo tutte e quattro le misure, **con gli stessi pinguini** nel training e nel test set.

> 🐍 `df.loc[indici, colonne]` seleziona le righe per **etichetta** (l'indice a sinistra della tabella) e le colonne per nome. Con gli indici di `X_train` e `X_test` teniamo gli stessi pinguini di prima. `.iloc[0]` invece prende la riga in **posizione** 0, cioè la prima.

Prima di addestrare il KNN guardiamo come è fatta la **distanza** tra due pinguini, misura per misura:

In [ ]:
misure = ["becco_lunghezza", "becco_profondita", "pinna", "peso"]
X4_train = df.loc[X_train.index, misure]     # stessi pinguini di X_train, con 4 colonne
X4_test = df.loc[X_test.index, misure]       # stessi pinguini di X_test (resta chiuso 🔒)

a = X4_train.iloc[0]                         # il primo pinguino del training set
b = X4_train.iloc[1]                         # il secondo
contributo = pd.DataFrame({"pinguino A": a, "pinguino B": b})
contributo["differenza al quadrato"] = (a - b) ** 2
# quanta parte della somma sotto la radice viene da ogni misura
contributo["quota"] = contributo["differenza al quadrato"] / contributo["differenza al quadrato"].sum()

print(f"Distanza tra i due pinguini: {contributo['differenza al quadrato'].sum() ** 0.5:.1f}")
contributo.round(4)

Il peso è in **grammi**, le altre misure in **millimetri**: la differenza di peso al quadrato vale 9 milioni, quella della pinna circa mille. La distanza è fatta al 99.99% dal peso: per questo KNN due pinguini sono "vicini" se pesano uguale, e il becco e la pinna non contano quasi nulla.

Il rimedio è la **scalatura**: `StandardScaler` trasforma ogni colonna in $z = \frac{x - \text{media}}{\text{deviazione standard}}$, così tutte hanno media 0 e deviazione standard 1 e pesano allo stesso modo nella distanza.

> 🐍 `make_pipeline(passo1, passo2)` incolla più passi in **un solo modello**: con `.fit` addestra lo scaler e poi il KNN sui dati scalati; con `.predict` scala i dati nuovi con la **stessa** media e deviazione standard e li passa al KNN.

Lo scaler va messo **dentro** la pipeline, e non applicato a tutta la tabella prima di cominciare. Così, in ogni giro della cross-validation, media e deviazione standard vengono calcolate **solo sui fold di training**: il fold di validazione resta un insieme di dati mai visti, com'è giusto che sia.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

knn_grezzo = KNeighborsClassifier(n_neighbors=5)                                    # senza scalatura
knn_scalato = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))  # scaler + KNN

print("Con le 4 misure:")
print(f"  KNN senza scalatura – accuracy in CV: {cross_val_score(knn_grezzo, X4_train, y_train, cv=cv).mean():.2f}")
print(f"  KNN con scalatura   – accuracy in CV: {cross_val_score(knn_scalato, X4_train, y_train, cv=cv).mean():.2f}")
print("Con il solo becco:")
print(f"  KNN senza scalatura – accuracy in CV: {cross_val_score(knn_grezzo, X_train, y_train, cv=cv).mean():.2f}")
print(f"  KNN con scalatura   – accuracy in CV: {cross_val_score(knn_scalato, X_train, y_train, cv=cv).mean():.2f}")

📌 **Gli algoritmi basati sulle distanze vogliono feature sulla stessa scala.** Con le quattro misure, senza scalatura il KNN fa 0.77: peggio che con il solo becco (0.82), perché in pratica guarda solo il peso. Con la scalatura sale a 0.91. Sul solo becco invece la scalatura non cambia il risultato (0.82 in entrambi i casi): le due misure sono entrambe in millimetri e su scale simili.

Gli alberi non hanno questo problema: ogni domanda confronta **una sola feature** con una soglia, e cambiare unità di misura sposta la soglia ma non la divisione.

### 2.4 ✏️ Esercizio: scegliere k con `GridSearchCV`
Scegliamo $k$ con `GridSearchCV`, come abbiamo fatto per l'albero. C'è una sola novità: il modello è una pipeline, e bisogna dire **a quale passo** appartiene l'iperparametro.

> 🐍 Dentro una pipeline creata con `make_pipeline` ogni passo prende il nome della sua classe in minuscolo: `standardscaler` e `kneighborsclassifier`. Nella griglia la chiave si scrive `"nomedelpasso__iperparametro"`, con **due** trattini bassi: per esempio `"kneighborsclassifier__n_neighbors"`.

Scrivi la griglia per provare i valori dispari di $k$ da 1 a 31 (`list(range(1, 32, 2))`).

In [ ]:
# ✏️ Esercizio: scrivi la griglia (con {} viene provato solo il KNN così com'è, con k = 5)
griglia_knn = {}

ricerca_knn = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), griglia_knn, cv=cv)
ricerca_knn.fit(X4_train, y_train)
print("Migliore:", ricerca_knn.best_params_, f"→ accuracy in CV {ricerca_knn.best_score_:.2f}")

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
griglia_knn = {"kneighborsclassifier__n_neighbors": list(range(1, 32, 2))}
```

La ricerca sceglie $k = 7$, con accuracy in cross-validation 0.91.

Il KNN ha anche altri iperparametri. Se vuoi, aggiungili alla griglia: `weights` decide se i vicini votano tutti allo stesso modo (`"uniform"`) o se i più vicini contano di più (`"distance"`), `metric` come si misura la distanza (`"euclidean"` o `"manhattan"`, la somma delle differenze in valore assoluto).

```python
griglia_knn = {
    "kneighborsclassifier__n_neighbors": list(range(1, 32, 2)),
    "kneighborsclassifier__weights": ["uniform", "distance"],
    "kneighborsclassifier__metric": ["euclidean", "manhattan"],
}
```

Con questa griglia vince `metric="manhattan"`, `n_neighbors=9`, `weights="uniform"`, con 0.92: alla pari con 0.91.

</details>

### 2.5 ✏️ Esercizio: il benchmark
Abbiamo quattro modelli: albero, random forest, gradient boosting e KNN. Quale conviene, usando tutte e quattro le misure? Confrontali **con la stessa cross-validation** `cv` per tutti: se ogni modello venisse valutato su fold diversi, una parte della differenza dipenderebbe dai fold e non dal modello. Con gli stessi fold il confronto è alla pari.

Per ogni modello ci servono la media e la deviazione standard dell'accuracy e il **tempo di addestramento**. Usiamo `cross_validate`, che restituisce un dizionario con un valore per ogni fold:
- `risultati["test_score"]`: le 5 accuracy sui fold di validazione;
- `risultati["fit_time"]`: i 5 tempi di addestramento, in secondi.

Completa la cella:
1. nel dizionario `modelli` sostituisci ogni `None` con il modello, usando gli iperparametri che abbiamo già visto: l'albero con `max_depth=5, min_samples_leaf=10`, la foresta con `n_estimators=200`, il gradient boosting con i valori di default, il KNN **in pipeline con lo scaler** e con il $k$ trovato nella 2.4 (per albero, foresta e boosting metti `random_state=0`);
2. nel ciclo calcola `cross_validate(modello, X4_train, y_train, cv=cv)`, salva i 5 punteggi in `punteggi_fold[nome]` e aggiungi a `righe` un dizionario con il nome del modello, l'accuracy media, la deviazione standard e il tempo medio di addestramento in millisecondi.

In [ ]:
# ✏️ Esercizio
from sklearn.model_selection import cross_validate

modelli = {
    "albero": None,      # ✏️ sostituisci None con il modello
    "foresta": None,
    "boosting": None,
    "knn": None,
}

righe = []               # una riga della tabella per ogni modello
punteggi_fold = {}       # nome del modello → i suoi 5 punteggi (servono per il grafico)
for nome, modello in modelli.items():        # .items(): le coppie (chiave, valore) del dizionario
    if modello is None:
        continue         # salta i modelli non ancora scritti: la cella funziona anche così
    # ✏️ risultati = cross_validate(...) con cv=cv sul training set con le 4 misure
    # ✏️ punteggi_fold[nome] = i 5 punteggi di validazione
    # ✏️ aggiungi a righe un dizionario con le chiavi:
    #    "modello", "accuracy media", "deviazione std", "fit (ms)"
    pass

benchmark = pd.DataFrame(righe)
benchmark.round(3)

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
modelli = {
    "albero": DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=0),
    "foresta": RandomForestClassifier(n_estimators=200, random_state=0),
    "boosting": GradientBoostingClassifier(random_state=0),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
}

righe = []
punteggi_fold = {}
for nome, modello in modelli.items():
    if modello is None:
        continue
    risultati = cross_validate(modello, X4_train, y_train, cv=cv)
    punteggi_fold[nome] = risultati["test_score"]
    righe.append({"modello": nome,
                  "accuracy media": risultati["test_score"].mean(),
                  "deviazione std": risultati["test_score"].std(),
                  "fit (ms)": risultati["fit_time"].mean() * 1000})     # da secondi a millisecondi

# il resto della cella (import e ultime due righe) resta com'è
```

</details>

Il grafico mostra i 5 punteggi di ogni modello (un punto per fold) e la loro media (il trattino nero):

In [ ]:
plt.figure(figsize=(8, 4.5))
for posizione, (nome, punteggi) in enumerate(punteggi_fold.items()):
    plt.scatter([posizione] * len(punteggi), punteggi, s=60, alpha=0.7, edgecolors="k")     # un punto per fold
    plt.hlines(punteggi.mean(), posizione - 0.25, posizione + 0.25, color="k", linewidth=2)  # la media
plt.xticks(range(len(punteggi_fold)), list(punteggi_fold.keys()))
plt.xlim(-0.6, max(len(punteggi_fold), 1) - 0.4)
plt.ylabel("accuracy sul fold di validazione"); plt.grid(alpha=0.3)
plt.title("Benchmark: i 5 fold di ogni modello (stessi fold per tutti)")
plt.show()

<details>
<summary><b>🤔 Come si legge il benchmark</b> (apri dopo aver fatto l'esercizio)</summary>

- **Sono tutti alla pari.** Le medie vanno da 0.89 a 0.91, e le deviazioni standard da 0.02 a 0.04: la differenza tra il primo e l'ultimo sta dentro la deviazione standard. Nel grafico i punti dei quattro modelli si sovrappongono.
- **Il salto vero l'hanno fatto le feature, non il modello.** Con il solo becco l'albero scelto si fermava a 0.82; lo stesso albero con le quattro misure arriva a 0.89.
- **Il costo non è lo stesso.** I tempi cambiano da un computer all'altro, ma non il loro ordine: albero e KNN si addestrano in pochissimo, mentre il boosting e soprattutto la foresta impiegano decine di volte tanto, per la stessa accuracy.
- Se il tuo KNN è intorno a 0.77, hai dimenticato la scalatura: rileggi la 2.3.

</details>

🔓 Il modello è scelto: prendiamo quello con l'accuracy media più alta in cross-validation, lo addestriamo su tutto il training set e **apriamo il test set**.

In [ ]:
if len(benchmark) > 0:
    migliore = benchmark.loc[benchmark["accuracy media"].idxmax()]     # la riga con la media più alta
    modello_scelto = modelli[migliore["modello"]].fit(X4_train, y_train)
    print(f"Modello scelto: {migliore['modello']} (accuracy in CV {migliore['accuracy media']:.2f})")
    print(f"🔓 Accuracy sul test set: {accuracy_score(y_test, modello_scelto.predict(X4_test)):.2f}")
else:
    print("Il benchmark è vuoto: completa l'esercizio 2.5 (o copia la soluzione) e riesegui questa cella.")

Con i modelli della soluzione viene scelto il KNN ($k = 7$, con scalatura): accuracy in cross-validation 0.91, sul test set **0.90**. La stima della cross-validation era onesta.

📌 **Il test set si apre una volta sola, sul modello scelto con la cross-validation.** Non abbiamo guardato il test set degli altri tre modelli: se lo facessimo e poi tenessimo il migliore, staremmo scegliendo con il test set. Nell'Esercitazione 2 questo stesso test set era già servito a valutare l'albero addestrato sul becco: quello era un altro esperimento, con due feature. Qui le feature sono quattro e il modello finale è uno solo.

### 2.6 ⏩ KNN per la regressione
Il KNN fa anche **regressione**: invece di far votare i $k$ vicini, ne fa la **media**. Riprendiamo il problema dell'Esercitazione 2, stimare il **peso** dalla lunghezza della **pinna**, con la stessa divisione in training e test set, e confrontiamo `KNeighborsRegressor` con l'albero di regressione con `max_depth=3`. Come nell'Esercitazione 2, qui il test set serve solo a osservare.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

X_peso = df[["pinna"]]          # una sola feature: qui la scalatura non serve
y_peso = df["peso"]
X_peso_train, X_peso_test, y_peso_train, y_peso_test = train_test_split(X_peso, y_peso, test_size=0.3, random_state=0)
pinne = pd.DataFrame({"pinna": np.linspace(150, 260, 500)})    # anche FUORI dal range dei dati (172-231 mm)

plt.figure(figsize=(10, 5.5))
plt.axvspan(150, X_peso["pinna"].min(), color="gold", alpha=0.15)      # zone senza dati
plt.axvspan(X_peso["pinna"].max(), 260, color="gold", alpha=0.15)
plt.scatter(X_peso_train["pinna"], y_peso_train, color="lightgray", edgecolors="k", linewidths=0.4, s=20)
for k_reg in [1, 5, 20]:
    knn_reg = KNeighborsRegressor(n_neighbors=k_reg).fit(X_peso_train, y_peso_train)
    mse = mean_squared_error(y_peso_test, knn_reg.predict(X_peso_test))
    plt.plot(pinne["pinna"], knn_reg.predict(pinne), linewidth=1.8, label=f"KNN, k = {k_reg} (MSE test {mse:.2f})")
albero_reg = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_peso_train, y_peso_train)
mse = mean_squared_error(y_peso_test, albero_reg.predict(X_peso_test))
plt.plot(pinne["pinna"], albero_reg.predict(pinne), color="k", linestyle="--", label=f"albero, max_depth = 3 (MSE test {mse:.2f})")
plt.xlim(150, 260); plt.xlabel("pinna (mm)"); plt.ylabel("peso (g)"); plt.grid(alpha=0.3)
plt.legend(fontsize=8, loc="upper left"); plt.title("KNN e albero di regressione: peso stimato dalla pinna")
plt.show()

Anche la curva del KNN è **a gradini**: la predizione cambia solo quando cambia l'insieme dei $k$ vicini. Con $k = 1$ insegue i singoli pinguini (MSE di test 277925.00), con $k = 20$ è molto più regolare (145149.88) ed è alla pari con l'albero (147642.35). E come l'albero, il KNN **non estrapola**: nelle zone gialle i vicini sono sempre gli stessi pinguini al bordo dei dati, e la predizione resta ferma.

---
# Parte 3 — Clustering: gruppi senza etichette

Finora ogni pinguino aveva un'etichetta da imparare (il sesso): era **apprendimento supervisionato**. Ora facciamo finta di non avere nessuna etichetta e chiediamo all'algoritmo di trovare da solo dei **gruppi** di pinguini simili: è il **clustering**, un problema **non supervisionato**.

> 📋 **Il problema: ci sono gruppi tra i pinguini?**
> - **Dati:** tutti i 333 pinguini
> - **Target `y`:** **nessuno**
> - **Feature `X`:** `becco_lunghezza` e `becco_profondita` (dalla 3.7 tutte e quattro le misure)
> - **Tipo di problema:** **clustering**
> - **Come lo valutiamo:** con misure che guardano solo la forma dei gruppi (inertia, silhouette). Alla fine, visto che noi la specie la conosciamo, controlleremo se i gruppi le corrispondono.

Qui **non serve dividere in training e test set**: non c'è un target da prevedere, quindi non c'è una risposta giusta da tenere nascosta per la valutazione. Usiamo tutti i pinguini.

### 3.1 Togliamo la specie
Riusiamo la tabella `X` della Parte 1, che contiene le due misure del becco di tutti i 333 pinguini, e la disegniamo senza colori.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X["becco_lunghezza"], X["becco_profondita"], color="lightgray", edgecolors="k", linewidths=0.5, s=35)
plt.xlabel("becco_lunghezza"); plt.ylabel("becco_profondita"); plt.grid(alpha=0.3)
plt.title("I 333 pinguini, senza etichette")
plt.show()

> 🤔 Quanti gruppi vedi? Dove metteresti i confini?

### 3.2 ✏️ Esercizio: fai tu un passo di k-means
**k-means** cerca $k$ gruppi, e $k$ lo scegliamo noi. Ogni gruppo è rappresentato dal suo **centroide**, un punto che fa da "centro" del gruppo. L'algoritmo parte da $k$ centroidi iniziali e ripete due passi:
1. **assegnazione**: ogni pinguino va nel gruppo del centroide **più vicino**;
2. **aggiornamento**: ogni centroide si sposta nella **media** dei pinguini del suo gruppo.

Si ferma quando nessun pinguino cambia più gruppo.

k-means usa le distanze, quindi come per il KNN prima **scaliamo** le due misure. Scegliamo $k = 3$ e partiamo apposta da tre centroidi iniziali sfortunati, tutti vicini al bordo sinistro.

> 🐍 `StandardScaler().fit_transform(X)` calcola media e deviazione standard di ogni colonna e restituisce i dati scalati, come **array NumPy** (una tabella di soli numeri, senza nomi di colonna). `np.array([[...], [...]])` crea un array a mano: qui una riga per centroide. `pd.DataFrame(Z, columns=[...])` trasforma l'array in una tabella di pandas, dando un nome alle colonne.

In [ ]:
Z = StandardScaler().fit_transform(X)        # il becco scalato: un array con 333 righe e 2 colonne

centroidi_iniziali = np.array([[-2.0, 1.0],  # centroide 0: (lunghezza, profondità) scalate
                               [-1.8, 0.5],  # centroide 1
                               [-1.5, 1.5]]) # centroide 2

passo = pd.DataFrame(Z, columns=["lunghezza", "profondita"])     # il becco scalato come tabella
passo.head()

Prima di far lavorare l'algoritmo, **fai tu il primo passo**, con gli stessi strumenti del KNN a mano della 2.1.

1. **Assegnazione.** Per ogni pinguino servono tre distanze, una per centroide. La colonna `d0` (distanza dal centroide 0) è già scritta: scrivi `d1` e `d2` allo stesso modo, con le coordinate degli altri due centroidi. La riga successiva, già pronta, mette in `cluster` il numero del centroide più vicino.
2. **Aggiornamento.** Calcola i nuovi centroidi: la media di `lunghezza` e `profondita` dei pinguini di ogni cluster. Serve `.groupby("cluster")`, che divide la tabella in gruppi, seguito dalle due colonne che interessano e da `.mean()`.

> 🐍 `np.argmin(tabella, axis=1)` guarda la tabella **una riga alla volta** e dice in quale colonna (0, 1 o 2) si trova il valore più piccolo. `.to_numpy()` trasforma le colonne di pandas in un array NumPy.

In [ ]:
# ✏️ Esercizio
# ASSEGNAZIONE: la distanza di ogni pinguino da ciascuno dei tre centroidi
passo["d0"] = ((passo["lunghezza"] - (-2.0)) ** 2 + (passo["profondita"] - 1.0) ** 2) ** 0.5   # dal centroide 0
passo["d1"] = 0      # ✏️ sostituisci 0 con la distanza dal centroide 1, che si trova in (-1.8, 0.5)
passo["d2"] = 0      # ✏️ sostituisci 0 con la distanza dal centroide 2, che si trova in (-1.5, 1.5)

# il cluster di ogni pinguino è il numero del centroide più vicino (già scritto)
passo["cluster"] = np.argmin(passo[["d0", "d1", "d2"]].to_numpy(), axis=1)
print(passo["cluster"].value_counts())       # quanti pinguini in ogni cluster

# AGGIORNAMENTO: ogni centroide si sposta nella media dei pinguini del suo cluster
# ✏️ sostituisci la riga qui sotto: raggruppa `passo` per "cluster", tieni le colonne
#    ["lunghezza", "profondita"] e calcola la media (per ora i centroidi restano quelli iniziali)
nuovi_centroidi = pd.DataFrame(centroidi_iniziali, columns=["lunghezza", "profondita"])
nuovi_centroidi

<details>
<summary><b>💡 Soluzione</b> (clicca per aprire, ma prima prova da solo!)</summary>

```python
passo["d1"] = ((passo["lunghezza"] - (-1.8)) ** 2 + (passo["profondita"] - 0.5) ** 2) ** 0.5   # dal centroide 1
passo["d2"] = ((passo["lunghezza"] - (-1.5)) ** 2 + (passo["profondita"] - 1.5) ** 2) ** 0.5   # dal centroide 2

nuovi_centroidi = passo.groupby("cluster")[["lunghezza", "profondita"]].mean()

# il resto della cella resta com'è
```

</details>

La funzione di supporto `passi_kmeans` esegue l'algoritmo completo e ne salva tutti i passi. Confrontiamo il suo secondo passo (il primo aggiornamento) con i tuoi centroidi:

In [ ]:
stati = passi_kmeans(Z, centroidi_iniziali)   # la lista di tutti i passi, dall'inizio alla convergenza

attesi = stati[2]["centroidi"]                # i centroidi dopo la prima assegnazione e il primo aggiornamento
print("Centroidi dopo il primo aggiornamento:")
print(pd.DataFrame(attesi, columns=["lunghezza", "profondita"]).round(2))

# np.allclose: True se i numeri coincidono (a meno di arrotondamenti)
if nuovi_centroidi.shape == attesi.shape and np.allclose(nuovi_centroidi.to_numpy(), attesi):
    print("✅ I tuoi centroidi coincidono: hai fatto un passo di k-means.")
else:
    print("❌ I tuoi centroidi non coincidono ancora: controlla d1, d2 e nuovi_centroidi.")

Dopo la prima assegnazione i cluster sono molto sbilanciati: al centroide 0 toccano solo 2 pinguini, al centroide 1 ne toccano 224. Con l'aggiornamento i centroidi 1 e 2 si spostano parecchio verso destra, in mezzo ai loro pinguini. Un solo passo non basta: l'algoritmo ripete assegnazione e aggiornamento finché i cluster smettono di cambiare.

### 3.3 k-means passo per passo
Ora guardiamo tutti i passi. Usa lo slider per avanzare di un passo alla volta: le ✕ sono i centroidi, le frecce mostrano di quanto si sono spostati. I passi 1 e 2 sono quelli che hai appena fatto a mano.

Nel titolo c'è l'**inertia**, che misura la qualità del risultato: è la somma delle distanze al quadrato tra ogni pinguino e il suo centroide. Più è bassa, più i gruppi sono compatti.

In [ ]:
print("Passi fino alla convergenza:", len(stati) - 1)

fig_km, ax_km = plt.subplots(figsize=(6.5, 4.8))
plt.close(fig_km)

def mostra_passo(numero):
    """Disegna lo stato di k-means dopo `numero` passi."""
    disegna_passo_kmeans(stati[numero], ax_km, passo=numero,
                         nomi=("becco_lunghezza (scalata)", "becco_profondita (scalata)"))
    return fig_km

slider_passo = IntSlider(min=0, max=len(stati) - 1, step=1, value=0, description="passo",
                         continuous_update=True, layout=Layout(width="600px"))
immagine_km = Image(format="png", layout=Layout(width="600px"))

def aggiorna_km(cambiamento=None):
    memoria = io.BytesIO()
    mostra_passo(slider_passo.value).savefig(memoria, format="png", bbox_inches="tight")
    immagine_km.value = memoria.getvalue()

slider_passo.observe(aggiorna_km, names="value")
display(VBox([slider_passo, immagine_km]))
aggiorna_km()

*Se lo slider non compare, puoi chiamare la funzione a mano, per esempio `mostra_passo(3)`.*

Guarda l'inertia nel titolo: a ogni passo **scende o resta uguale, non sale mai**. L'assegnazione avvicina ogni pinguino al suo centroide, l'aggiornamento mette ogni centroide nel punto che minimizza le distanze dai suoi pinguini. Per questo k-means si ferma sempre.

📌 **Il risultato dipende dai centroidi iniziali.** k-means si ferma quando non riesce più a migliorare, ma non è detto che abbia trovato i gruppi migliori in assoluto: con un'altra partenza può fermarsi su gruppi diversi. Per questo scikit-learn sceglie i centroidi iniziali lontani tra loro (il metodo **k-means++**, il default) e con `n_init=10` ripete tutto da 10 partenze diverse, tenendo il risultato con l'inertia più bassa.

### 3.4 `KMeans` in scikit-learn
`KMeans` si usa come gli altri modelli, in pipeline con lo scaler. La differenza è che a `fit` passiamo **solo `X`**: non c'è nessuna `y`. `fit_predict(X)` trova i cluster e restituisce, per ogni pinguino, il **numero** del suo cluster (0, 1 o 2).

> 🐍 Una pipeline si può indicizzare come una lista: `kmeans[0]` è il primo passo (lo scaler), `kmeans[1]` il secondo (il `KMeans` vero e proprio). `scaler.inverse_transform(...)` fa la trasformazione inversa della scalatura: riporta i valori in millimetri.

Accanto mettiamo k-means **senza scalatura**, sugli stessi dati.

In [ ]:
from sklearn.cluster import KMeans

kmeans = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=0))
cluster = kmeans.fit_predict(X)                 # solo X: nessun target. Un numero di cluster per ogni pinguino
print("Cluster dei primi 10 pinguini:", cluster[:10])

# i centroidi sono nello spazio scalato: li riportiamo in millimetri
centroidi_mm = kmeans[0].inverse_transform(kmeans[1].cluster_centers_)
print("\nCentroidi (in mm):")
print(pd.DataFrame(centroidi_mm, columns=X.columns).round(1))

kmeans_grezzo = KMeans(n_clusters=3, n_init=10, random_state=0)      # senza scaler
cluster_grezzo = kmeans_grezzo.fit_predict(X)
print("\nDeviazione standard delle due misure (in mm):")
print(X.std().round(2))

fig, assi = plt.subplots(1, 2, figsize=(15, 5))
mostra_cluster(X, cluster, centroidi_mm, "k-means con scalatura", ax=assi[0])
mostra_cluster(X, cluster_grezzo, kmeans_grezzo.cluster_centers_, "k-means senza scalatura", ax=assi[1])
plt.show()

Il **centroide** è la media dei pinguini del suo cluster: per esempio un becco lungo 38.8 mm e profondo 18.3 mm. **Non è un pinguino reale**, ma il "pinguino medio" del gruppo. Il numero di ogni cluster (e quindi il suo colore) è arbitrario: conta solo quali pinguini stanno insieme.

Senza scalatura i cluster **cambiano**: i confini diventano quasi verticali, e il gruppo in basso viene spezzato in due. Le due misure sono entrambe in millimetri, ma la lunghezza varia molto più della profondità (deviazione standard 5.47 mm contro 1.97 mm), quindi nelle distanze conta quasi solo la lunghezza. È lo stesso problema della 2.3: con le distanze, la scala delle feature decide il risultato.

### 3.5 Scegliere k: il gomito e la silhouette
Abbiamo chiesto 3 cluster, ma chi l'ha detto che sono 3? Senza etichette non c'è un'accuracy da calcolare. Ci sono due strumenti, che guardano solo la forma dei gruppi.

**Il gomito.** L'inertia scende **sempre** aumentando $k$ (con un cluster per pinguino sarebbe 0), quindi non si può scegliere il $k$ con l'inertia più bassa. Si cerca invece il "gomito": il punto dopo il quale aggiungere cluster fa guadagnare poco.

**La silhouette.** Per ogni pinguino si calcolano:
- $a$: la distanza media dagli altri pinguini **del suo cluster**;
- $b$: la distanza media dai pinguini del cluster **vicino più prossimo**.

$$s = \frac{b - a}{\max(a, b)}$$

$s$ va da $-1$ a $1$: vicino a 1 se il pinguino è ben dentro il suo cluster ($a$ piccola, $b$ grande), vicino a 0 se sta sul confine tra due cluster, negativo se è più vicino a un altro cluster che al suo. `silhouette_score` restituisce la media su tutti i pinguini: **più è alta, meglio è**.

Proviamo $k$ da 2 a 7 sui dati scalati `Z` della 3.2: per ogni $k$ addestriamo un `KMeans` e salviamo la sua inertia e la sua silhouette.

In [ ]:
from sklearn.metrics import silhouette_score

valori_k = list(range(2, 8))          # k da 2 a 7
inerzie, silhouette = [], []          # due liste vuote: le riempiamo nel ciclo

for k in valori_k:
    modello = KMeans(n_clusters=k, n_init=10, random_state=0)    # k-means con k cluster
    modello.fit(Z)                                               # solo i dati: nessuna y
    inerzie.append(modello.inertia_)                             # .inertia_: l'inertia finale
    # .labels_: il cluster assegnato a ogni pinguino; silhouette_score vuole i dati e i cluster
    silhouette.append(silhouette_score(Z, modello.labels_))

print("Silhouette:", dict(zip(valori_k, np.round(silhouette, 2).tolist())))   # zip accoppia le due liste

fig, assi = plt.subplots(1, 2, figsize=(13, 4.2))
assi[0].plot(valori_k, inerzie, marker="o")
assi[0].set_title("Il gomito: inertia al variare di k"); assi[0].set_ylabel("inertia")
assi[1].plot(valori_k, silhouette, marker="o", color="crimson")
assi[1].set_title("Silhouette media al variare di k"); assi[1].set_ylabel("silhouette")
for ax in assi:
    ax.set_xlabel("k (numero di cluster)"); ax.set_xticks(valori_k); ax.grid(alpha=0.3)
plt.show()

L'inertia scende sempre, ma il calo più forte è tra $k = 2$ e $k = 3$: da 3 in poi la curva si appiattisce, e il gomito è a $k = 3$. La silhouette dice la stessa cosa in modo più netto: 0.47 con $k = 2$, **0.52 con $k = 3$** (il massimo), 0.44 con $k = 4$. Sul becco, i due strumenti scelgono 3 cluster.

### 3.6 Confronto con le specie vere
In un vero problema di clustering le etichette non ci sono. Qui però la specie la conosciamo, e possiamo controllare se i 3 cluster della 3.4 le corrispondono.

> 🐍 `pd.crosstab(a, b)` conta quante volte compare ogni coppia di valori: una riga per ogni valore di `a`, una colonna per ogni valore di `b`. È lo stesso tipo di tabella della matrice di confusione.

In [ ]:
from sklearn.metrics import adjusted_rand_score

print(f"ARI tra cluster e specie: {adjusted_rand_score(df['specie'], cluster):.2f}")
pd.crosstab(cluster, df["specie"], rownames=["cluster"])     # righe: i cluster di k-means; colonne: la specie vera

Ogni cluster corrisponde quasi a una specie: uno raccoglie 142 dei 146 Adelie, uno 112 dei 119 Gentoo, uno 54 dei 68 Chinstrap. La specie trovata peggio è la **Chinstrap**: 9 finiscono nel cluster dei Gentoo e 5 in quello degli Adelie. Nel grafico della 3.4 è il gruppo in alto a destra, quello con i confini meno netti.

L'**ARI** (*Adjusted Rand Index*) riassume la tabella in un numero: vale 1 se cluster e specie coincidono (a meno dei nomi dei cluster) e circa 0 se i cluster sono assegnati a caso. Qui vale 0.82.

📌 L'ARI è una **metrica esterna**: per calcolarla servono le etichette vere, che in un problema di clustering di solito **non abbiamo**. Inertia e silhouette invece sono metriche **interne**: usano solo i dati e i cluster.

### 3.7 Con tutte le misure
Ripetiamo la ricerca di $k$ usando tutte e quattro le misure, scalate.

In [ ]:
Z4 = StandardScaler().fit_transform(df[misure])       # le 4 misure scalate, per tutti i 333 pinguini

righe = []
for k in range(2, 8):
    modello = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z4)
    righe.append({"k": k,
                  "silhouette": silhouette_score(Z4, modello.labels_),
                  "ARI con le specie": adjusted_rand_score(df["specie"], modello.labels_)})
print(pd.DataFrame(righe).round(2).to_string(index=False))     # .to_string(index=False): stampa senza l'indice

cluster_2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Z4)
pd.crosstab(cluster_2, df["specie"], rownames=["cluster (k = 2)"])

Con le quattro misure la silhouette preferisce **$k = 2$** (0.53, contro 0.45 con $k = 3$). La tabella mostra quali sono i due gruppi: tutti i Gentoo da una parte, Adelie e Chinstrap insieme dall'altra. Dal punto di vista della forma dei gruppi è una risposta sensata: i Gentoo sono molto diversi dalle altre due specie per pinna e peso. Eppure le specie sono 3, e con $k = 3$ l'ARI è più alta (0.80 contro 0.65).

📌 **Nessuna metrica sostituisce l'interpretazione.** La silhouette premia i gruppi ben separati, e non sa quale domanda ci interessa. I cluster vanno guardati e devono **avere senso per il problema**: il valore di $k$ si sceglie anche in base a quello che dei gruppi si vuole fare.

### 3.8 ⏩ Altri algoritmi di clustering
k-means non è l'unico algoritmo. Il **clustering gerarchico** non chiede di fissare $k$ in partenza: parte con un cluster per ogni punto e a ogni passo unisce i due cluster più vicini, fino ad averne uno solo. Il risultato è un albero, il **dendrogramma**: più in alto avviene un'unione, più i due gruppi uniti erano lontani. Tagliandolo a una certa altezza si ottengono i cluster. Lo disegniamo su 40 pinguini estratti a caso (con tutti e 333 le etichette non si leggerebbero), usando le quattro misure scalate.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram     # scipy: libreria scientifica, installata insieme a scikit-learn

campione = df.sample(40, random_state=0)                    # 40 pinguini estratti a caso
Z_campione = StandardScaler().fit_transform(campione[misure])

unioni = linkage(Z_campione, method="ward")                 # "ward": unisce i due cluster che fanno crescere meno l'inertia
plt.figure(figsize=(13, 4.5))
dendrogram(unioni, labels=campione["specie"].tolist(), leaf_rotation=90, leaf_font_size=8, color_threshold=0,
           above_threshold_color="gray")
plt.ylabel("distanza tra i gruppi uniti"); plt.title("Dendrogramma di 40 pinguini (la specie serve solo come etichetta)")
plt.show()

L'ultima unione, la più alta, separa i Gentoo da tutti gli altri: è la stessa divisione in due trovata dalla silhouette nella 3.7.

| Algoritmo | Idea | Quando serve |
|---|---|---|
| **k-means** | $k$ centroidi, ogni punto va al più vicino | gruppi compatti e di dimensioni simili; $k$ va scelto prima; veloce |
| **Gerarchico** | unisce via via i gruppi più vicini (dendrogramma) | per vedere la struttura a più livelli e scegliere $k$ dopo; lento con tanti dati |
| **GMM** (misture di gaussiane) | ogni cluster è una "nuvola" gaussiana; ogni punto ha una **probabilità** di appartenere a ogni cluster | gruppi allungati o di dimensioni diverse; quando serve un'appartenenza sfumata |
| **DBSCAN** | un cluster è una zona **densa** di punti; i punti isolati sono **rumore** | gruppi di forma qualsiasi, presenza di outlier; non richiede $k$ |

In scikit-learn: `AgglomerativeClustering`, `GaussianMixture`, `DBSCAN`.

---
# Riepilogo

| Concetto | Dove l'abbiamo visto | In scikit-learn |
|---|---|---|
| Varianza di un albero, bias e varianza | 1.2 | campioni bootstrap |
| Bootstrap e out-of-bag | 1.3, 1.4 | `oob_score=True`, `oob_score_` |
| Random forest (bagging) | 1.4 | `RandomForestClassifier`, `n_estimators`, `max_features` |
| Boosting ed early stopping | 1.5 | `GradientBoostingClassifier`, `learning_rate`, `n_iter_no_change`, `validation_fraction` |
| AdaBoost | 1.6 ⏩ | `AdaBoostClassifier`, `sample_weight` |
| KNN | 2.1, 2.2 | `KNeighborsClassifier`, `n_neighbors` |
| Scalatura e pipeline | 2.3 | `StandardScaler`, `make_pipeline` |
| Iperparametri dentro una pipeline | 2.4 | `"passo__iperparametro"`, `weights`, `metric` |
| Confronto tra modelli sugli stessi fold | 2.5 | `cross_validate`: `test_score`, `fit_time` |
| Test set usato solo per il modello finale | 2.5 | 🔒 / 🔓 |
| KNN per la regressione | 2.6 ⏩ | `KNeighborsRegressor` |
| k-means | 3.2–3.4 | `KMeans`, `n_clusters`, `n_init`, `fit_predict`, `cluster_centers_`, `inertia_` |
| Scegliere k: gomito e silhouette | 3.5, 3.7 | `inertia_`, `silhouette_score` |
| Confronto con etichette note | 3.6 | `pd.crosstab`, `adjusted_rand_score` |
| Altri algoritmi di clustering | 3.8 ⏩ | `scipy.cluster.hierarchy`, `AgglomerativeClustering`, `GaussianMixture`, `DBSCAN` |

### KNN e k-means a confronto
Hanno nomi simili e usano entrambi le distanze, ma risolvono problemi diversi.

| | KNN | k-means |
|---|---|---|
| **Tipo** | supervisionato (classificazione o regressione) | non supervisionato (clustering) |
| **Dati** | servono le etichette `y` | solo `X`, nessuna etichetta |
| **Training** | nessuno: memorizza i dati (*lazy*) | iterativo: assegnazione e aggiornamento fino alla convergenza |
| **Che cos'è k** | il numero di **vicini** che votano | il numero di **cluster** da trovare |
| **Output** | la classe (o il valore) di un punto nuovo | il cluster di ogni punto e i centroidi |
| **In comune** | usano la **distanza**: le feature vanno **scalate** | usano la **distanza**: le feature vanno **scalate** |